# How many replicates does a check need?

Reads the seven replicates of one configuration written by

```bash
python experiments/exploration-replicate-variability/run.py
```

and turns them into the one number that decides exp-01's replicate count: the
standard deviation of the check's mean score across identical runs.

The note is
[`thinking/experiments/exploration-replicate-variability.md`](../../thinking/experiments/exploration-replicate-variability.md).
It records what this probe can and cannot support — read that before using
any number here to size an experiment.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

from soda_mmqc import cli

RUNS = Path("../../experiments/runs/exploration-replicate-variability").resolve()
CHECKLIST, CHECK, MODEL = "fig-checklist-exp01", "micrograph-scale-bar", "claude-sonnet-5"

reps = sorted((RUNS / "pinned").glob("rep-*"))
assert reps, f"no replicates under {RUNS}; run the probe first"
[p.name for p in reps]

## Score each replicate on its own

One `rep-NN/` is one predictions directory, which is exactly what
`score_check` takes. Seven calls, seven independent measurements of the same
configuration.

In [ ]:
rows = []
for rep in reps:
    index = int(rep.name.split("-")[1])
    result = cli.score_check(CHECKLIST, CHECK, rep, model=MODEL, save=False)
    for record in result["agentic"]["flat"]:
        analysis = record["analysis"]
        for prop, summary in (analysis.get("by_property") or {}).items():
            rows.append({
                "replicate": index,
                "example": record.get("doc_id"),
                "property": prop,
                "mean_score": summary.get("mean_score"),
                "layer1": tuple(sorted((summary.get("layer1_counts") or {}).items())),
            })

scores = pd.DataFrame(rows)
print(f"{len(reps)} replicates, {scores['example'].nunique()} examples, "
      f"{scores['property'].nunique()} properties")
scores.head()

## The number that decides it

The check's score for one replicate is the mean over its properties and
examples. Seven of those give the run-to-run spread.

`SE(n) = SD / sqrt(n)` is what a given replicate count buys. Compare it
against the difference exp-01 is trying to resolve: if the difference between
detailed and minimal skills is smaller than `SE(5)`, five replicates will not
find it and the design needs changing rather than more sampling.

In [ ]:
per_replicate = scores.groupby("replicate")["mean_score"].mean()
sd = per_replicate.std(ddof=1)

print(per_replicate.round(4).to_string())
print(f"\nmean over replicates : {per_replicate.mean():.4f}")
print(f"SD between replicates: {sd:.4f}")
print(f"range                : {per_replicate.max() - per_replicate.min():.4f}")

print("\nstandard error by replicate count:")
for n in (1, 3, 5, 7):
    print(f"  n={n}: SE = {sd/np.sqrt(n):.4f}")

## Where the variance lives

A check-level mean can be stable while individual examples swing. If the
spread is concentrated in a few examples or a few properties, that is worth
knowing: it may be a property of those cases rather than of the model, and it
changes whether more replicates help.

In [ ]:
by_example = (
    scores.groupby(["example", "replicate"])["mean_score"].mean()
    .groupby("example").agg(["mean", "std"])
    .sort_values("std", ascending=False)
)
by_example.head(10).round(4)

In [ ]:
by_property = (
    scores.groupby(["property", "replicate"])["mean_score"].mean()
    .groupby("property").agg(["mean", "std"])
    .sort_values("std", ascending=False)
)
by_property.round(4)

## Non-response

A replicate that returned nothing scores as a fully missing row set rather
than being excluded. Count them: a configuration that occasionally produces
nothing is unreliable in a way an average hides, and it bears directly on how
many replicates are needed.

In [ ]:
empty = []
for rep in reps:
    index = int(rep.name.split("-")[1])
    for prediction in sorted(rep.rglob("prediction.json")):
        import json as _json
        outputs = _json.loads(prediction.read_text())["outputs"]
        if not outputs:
            empty.append({"replicate": index,
                          "example": str(prediction.parent.relative_to(rep))})

pd.DataFrame(empty) if empty else "no empty answers in any replicate"

## Cost

Every session records `total_cost_usd`. This is what a replicate actually
costs, and what exp-01 would multiply by 11 checks and 2 arms.

In [ ]:
import json as _json

usage = []
for rep in reps:
    index = int(rep.name.split("-")[1])
    for audit in sorted(rep.rglob("tool_audit.json")):
        spent = _json.loads(audit.read_text()).get("usage") or {}
        if spent.get("total_cost_usd") is not None:
            usage.append({
                "replicate": index,
                "usd": spent["total_cost_usd"],
                "turns": spent.get("num_turns"),
                "seconds": (spent.get("duration_ms") or 0) / 1000,
            })

spend = pd.DataFrame(usage)
if len(spend):
    per_session = spend["usd"].mean()
    print(f"sessions       : {len(spend)}")
    print(f"per session    : ${per_session:.4f}")
    print(f"this probe     : ${spend['usd'].sum():.2f}")
    print(f"median turns   : {spend['turns'].median():.0f}")
    print(f"median seconds : {spend['seconds'].median():.1f}")
    print("\nexp-01 at 11 checks x 2 arms x 436 example-checks:")
    for n in (3, 5):
        print(f"  {n} replicates: {436*2*n:>5,} sessions, ~${436*2*n*per_session:,.0f}")
else:
    print("no usage recorded -- these runs predate cost capture")

## The decision

Write it in the note, not here. The note says what this probe can support:
it measures one arm of one check, and exp-01's statistic is a paired
difference whose variance is not this one. Treat `SE(n)` above as an order of
magnitude for sizing, and record in the note which count was chosen and why.